# project_25_capstone_dbtl — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [1]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

Python : 3.11.15
Platform: Linux-6.18.5-x86_64-with-glibc2.39
GPU    : NONE FOUND
Structure prediction on CPU is impractically slow.


## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [2]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

Note: you may need to restart the kernel to use updated packages.
Core install done.


In [3]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

# Environment stamp 2026-06-24T03:18:04 UTC
Bio            1.84
py3Dmol        2.4.0


numpy          2.4.6


pandas         3.0.3
matplotlib     3.11.0


seaborn        0.13.2
tqdm           4.68.3
requests       2.33.1


## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [4]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

Helpers ready: install_colabfold(), install_esmfold().


## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [5]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

seeds set to 0
logged: Ran 00_setup; environment stamped.


## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [6]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

Uncomment to mount Drive and set your working directory.


---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — choose the target, success criteria, controls, and the ML framing

**Standard slot:** *define & explore.* **For Project 25 (the capstone) this means:** make the two
decisions the whole campaign rests on — (1) **which real target** you will design against (any tier),
and (2) how you will **close the loop** by learning, from the cohort's accumulated design→outcome
data, which features actually predict success. You also state measurable success criteria + controls
and run a deterministic **mock / EXAMPLE_DATA** hello-world (D0).

> **This is the INTEGRATIVE capstone.** It reuses a full campaign workflow (here we use a **binder**
> as the worked example — but the *design_type is whichever you and your advisor choose*: binder,
> antibody, enzyme, monomer...) **and** adds the cohort-wide **ML success predictor** that is the
> project's signature deliverable.

> **RESPONSIBLE RESEARCH — read first.** Your **advisor MUST approve your chosen target against the
> `MASTER_BLUEPRINT.md §7` policy BEFORE you start the design campaign (P2).** Default any ambiguous
> target to a neutralizing / diagnostic / inhibitory / industrial framing. No out-of-scope targets
> (nothing that enhances pathogen transmissibility/virulence, toxins, screening evasion, or is meant
> to cause harm). See `README.md` / `MANUAL.md §8`.

Run `00_setup.ipynb` first in this session.

## The four non-negotiable messages (the capstone must embody all four)

1. **A computational design is a hypothesis, not a result.** A low `pae_interaction` / `scrmsd` is
   *confidence*, not measured binding or function.
2. **Diversity before filtering.** Generate many (100s–1000s), filter aggressively, never polish one.
3. **Controls are mandatory** — even in the *plan* (positive, scrambled-interface/dead-mutant
   negative, unrelated-protein negative).
4. **Report the hit rate, not the cherry.** Include the failures; report N pass / N generated.

The capstone adds a fifth, data-driven idea: **learn from the whole cohort which filters actually
work.** No single in-silico metric perfectly separates true from false hits — so we *train a model*
on accumulated design→outcome data and ask, honestly, whether it beats the field's single-metric
cutoffs.

## Setup paths

In [7]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_25_capstone_dbtl/notebooks


## 1 · Choose + justify your target (responsible-research check)

Fill in the block below for **your** advisor-approved target. The worked example is a **binder vs a
checkpoint protein** (the Project 06 PD-L1 pattern), but your `DESIGN_TYPE` is whatever you chose.

A defensible choice states: the protein, the *function* you want (neutralize / inhibit / diagnose /
catalyze an industrial reaction), the in-scope framing, and the structural data you will design
against. **Do not proceed to P2 until your advisor signs off the responsible-research check.**

In [8]:
# ---- EDIT THIS BLOCK FOR YOUR TARGET (advisor-approved) ----
TARGET_NAME   = "EXAMPLE_TARGET (worked example: a checkpoint protein, binder design)"
DESIGN_TYPE   = "binder"     # binder | antibody | enzyme | monomer  (YOUR choice; drives the filter cutoffs)
FRAMING       = "inhibitory / therapeutic-diagnostic (in scope under MASTER_BLUEPRINT §7)"
STRUCTURE_SRC = "RCSB accession(s) you verify in Week 1 (see data/README.md)"
ADVISOR_APPROVED = False     # <-- set True only AFTER your advisor signs the §7 responsible-research check

assert DESIGN_TYPE in {"binder", "antibody", "enzyme", "monomer"}, "pick a supported design_type"
print("target      :", TARGET_NAME)
print("design_type :", DESIGN_TYPE, " (drives the shared-filter cutoffs in notebook 03)")
print("framing     :", FRAMING)
print("advisor responsible-research approval recorded:", ADVISOR_APPROVED)
if not ADVISOR_APPROVED:
    print("\n>>> STOP before P2 (the design campaign) until ADVISOR_APPROVED is True. <<<")

target      : EXAMPLE_TARGET (worked example: a checkpoint protein, binder design)
design_type : binder  (drives the shared-filter cutoffs in notebook 03)
framing     : inhibitory / therapeutic-diagnostic (in scope under MASTER_BLUEPRINT §7)
advisor responsible-research approval recorded: False

>>> STOP before P2 (the design campaign) until ADVISOR_APPROVED is True. <<<


## 2 · Success criteria + controls (write them down NOW, not after)

State measurable criteria up front so you cannot move the goalposts later. Example for a binder:
"≥ N designs passing all binder layers with `pae_interaction ≤ 10` and `rosetta_dG ≤ −30`, with a
PD-1-footprint overlap ≥ 0.5"; for an enzyme: "≥ N designs with catalytic-geometry RMSD ≤ 0.5 Å."

Controls you commit to (used in the validation plan, notebook 05):
- **Positive:** a known-good binder/enzyme/natural protein (confirms the assay works).
- **Negative (scrambled-interface / catalytic dead-mutant):** your *own* top design, broken — it must
  lose activity. The cleanest specificity control.
- **Unrelated-protein negative.**

In [9]:
SUCCESS_CRITERIA = {
    "min_designs_all_layers": 5,          # >=5 designs passing every filter layer (EDIT for your target)
    "key_metric": "pae_interaction <= 10" if DESIGN_TYPE in ("binder", "antibody") else "catalytic_geom_rmsd <= 0.5",
    "novelty": "report TM-score to PDB (novel if < 0.5)",
}
CONTROLS = ["positive: known-good binder/enzyme/natural protein",
            "negative: scrambled-interface / catalytic dead-mutant (your own top design, broken)",
            "negative: unrelated protein of similar size"]
print("success criteria:")
for k, v in SUCCESS_CRITERIA.items():
    print(f"  {k}: {v}")
print("\ncontrols (committed up front):")
for c in CONTROLS:
    print("  -", c)

success criteria:
  min_designs_all_layers: 5
  key_metric: pae_interaction <= 10
  novelty: report TM-score to PDB (novel if < 0.5)

controls (committed up front):
  - positive: known-good binder/enzyme/natural protein
  - negative: scrambled-interface / catalytic dead-mutant (your own top design, broken)
  - negative: unrelated protein of similar size


## 3 · The ML framing — the capstone's signature

The campaign produces designs with in-silico metrics. The cohort (Projects 01–24) produced **many
more**, some with experimental labels. The capstone's distinctive question:

> *Given the cohort's accumulated `features → outcome` data, can we learn a predictor that triages
> designs better than the field's single-metric cutoffs (scRMSD<2, pae_interaction<10, ...)?*

`scripts/ml_predictor.py` is import-safe and deterministic. For teaching with no real cohort yet, it
builds a clearly-labeled **`EXAMPLE_DATA`** synthetic cohort (deterministic seed) with a *planted,
imperfect* feature→outcome structure — so this notebook runs anywhere. **Every synthetic row is
flagged `EXAMPLE_DATA`; never present these as real outcomes.**

In [10]:
import ml_predictor as ml

# Hello-world: build a tiny EXAMPLE_DATA cohort slice and peek at the schema.
demo = ml.build_cohort_table(n_per_target=20, seed=0)
print("EXAMPLE_DATA cohort slice:", demo.shape)
print("feature columns the predictor learns from:", ml.FEATURE_COLUMNS)
print("\nfield-standard single-metric cutoffs (the baselines to beat):")
for k, (op, thr) in ml.STANDARD_CUTOFFS.items():
    print(f"  {k} {op} {thr}")
print("\nsuccess-rate in this EXAMPLE_DATA slice (synthetic!):", round(demo["success"].mean(), 3))
demo.head(4)[["design_id", "design_type", "scrmsd", "pae_interaction", "plddt", "source", "success"]]

EXAMPLE_DATA cohort slice: (80, 13)
feature columns the predictor learns from: ['scrmsd', 'plddt', 'pae_interaction', 'solubility', 'rosetta_dG', 'shape_complementarity', 'tm_to_pdb']

field-standard single-metric cutoffs (the baselines to beat):
  scrmsd <= 2.0
  plddt >= 80.0
  pae_interaction <= 10.0
  rosetta_dG <= -30.0
  shape_complementarity >= 0.6

success-rate in this EXAMPLE_DATA slice (synthetic!): 0.175


,design_id,design_type,scrmsd,pae_interaction,plddt,source,success
0,EXAMPLE_DATA_binder_0_0000,binder,3.120,3.78,71.2,EXAMPLE_DATA,0
1,EXAMPLE_DATA_binder_0_0001,binder,4.253,3.05,89.7,EXAMPLE_DATA,0
2,EXAMPLE_DATA_binder_0_0002,binder,1.839,3.54,76.4,EXAMPLE_DATA,0
3,EXAMPLE_DATA_binder_0_0003,binder,4.427,15.36,85.3,EXAMPLE_DATA,0


## 4 · Mock hello-world: a tiny binder mini-run (worked example)

The campaign workflow (notebook 02) uses the binder-family pattern as the worked example. Here we run
the deterministic **mock** backend so the plumbing executes with no GPU. If your `DESIGN_TYPE` is an
enzyme/antibody, you will swap in that family's generator in notebook 02 — the *integration pattern*
(generate → score → cohort table → filter → ML) is identical. **Mock numbers are `SYNTHETIC`.**

In [11]:
import campaign_tools as ct

TARGET = "EXAMPLE_TARGET"
HOTSPOTS = ct.parse_hotspots("A56,A66,A115")   # EXAMPLE — replace with residues you derive from your target
designs = ct.generate_designs(TARGET, HOTSPOTS, n=4, design_type=DESIGN_TYPE, tool="mock")
ct.score_designs(designs, tool="mock")
d = designs[0]
print("example design:", d.design_id, "| len =", d.length, "aa")
print("  scrmsd =", d.scrmsd, " pae_interaction =", d.pae_interaction,
      " plddt =", d.plddt, " (SYNTHETIC)")
print("  synthetic flag:", d.synthetic, "->", d.notes[0])
print("\nSwitch tool='mock' -> the real backend (BindCraft/RFdiffusion/RFantibody/RFdiffusion2) on Colab (A100). See MANUAL.md §2.")

example design: EXAMPLE_DATA_binder_0000 | len = 40 aa
  scrmsd = 1.03  pae_interaction = 19.0  plddt = 83.0  (SYNTHETIC)
  synthetic flag: True -> SYNTHETIC — mock backend, not a real design/prediction (EXAMPLE_DATA)

Switch tool='mock' -> the real backend (BindCraft/RFdiffusion/RFantibody/RFdiffusion2) on Colab (A100). See MANUAL.md §2.


## D0 checklist
- [ ] **Target chosen + justified**; `DESIGN_TYPE` set; structural source noted (verify accessions Week 1).
- [ ] **Advisor responsible-research approval (§7) recorded** — required before P2.
- [ ] Measurable **success criteria** + **controls** written down up front.
- [ ] ML framing understood; `EXAMPLE_DATA` cohort slice inspected (flagged synthetic).
- [ ] Reproduced the mock mini-run (worked-example binder) with metrics printed and flagged SYNTHETIC.
- [ ] `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the full design campaign + assemble the cohort feature table.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — full design campaign on your target + assemble the cohort feature table

**Standard slot:** *design campaign.* **For Project 25 this is two things at once (D2):**
1. run a **complete design campaign** on your advisor-approved target (worked example: a binder
   campaign, the Project 06 pattern) at honest scale, and
2. **assemble the cohort feature table** — the `EXAMPLE_DATA` design→outcome dataset (with a planted
   feature→outcome structure) that the ML success predictor (notebook 04) learns from. In a real
   cohort this table is built from the Projects 01–24 outputs; here it is synthetic and clearly
   labeled.

> **Compute honesty:** a real campaign at this scale wants an **A100** (Colab Pro+ or a cluster); the
> capstone aggregates a full cohort's compute. **The ML part is light/CPU-OK** (scikit-learn/XGBoost
> on a feature table). The cells below run on the deterministic **mock / EXAMPLE_DATA** path so the
> plumbing executes anywhere; switch to the real backend on Colab. Run `00_setup.ipynb` first.

## Setup paths

In [12]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_25_capstone_dbtl/notebooks


## Version-verify the pinned upstreams (tools change!)

The capstone uses the **full stack** plus the ML libraries. **Pin commits/tags** and **verify the
URLs still exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin
and log it). This check needs no GPU. Which design tools you verify depends on your `DESIGN_TYPE`.

In [13]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   ML (always):
#     scikit-learn  https://github.com/scikit-learn/scikit-learn   # success predictor (LogReg/RF) — pin <tag>
#     XGBoost       https://github.com/dmlc/xgboost                 # optional gradient-boosted predictor — pin <tag>
#   Design tools (verify the ones YOUR campaign uses):
#     RFdiffusion   https://github.com/RosettaCommons/RFdiffusion   # backbones / binder mode — pin <commit>
#     BindCraft     https://github.com/martinpacesa/BindCraft       # one-shot binders (A100) — pin <commit>
#     ColabFold     https://github.com/sokrypton/ColabFold          # AF2(-Multimer) — pin <commit>
PINNED = {
    "scikit-learn": "https://github.com/scikit-learn/scikit-learn",
    "XGBoost":      "https://github.com/dmlc/xgboost",
    "RFdiffusion":  "https://github.com/RosettaCommons/RFdiffusion",
    "BindCraft":    "https://github.com/martinpacesa/BindCraft",
    "ColabFold":    "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:13s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:13s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")

  [200] scikit-learn  https://github.com/scikit-learn/scikit-learn


  [200] XGBoost       https://github.com/dmlc/xgboost


  [200] RFdiffusion   https://github.com/RosettaCommons/RFdiffusion


  [200] BindCraft     https://github.com/martinpacesa/BindCraft


  [200] ColabFold     https://github.com/sokrypton/ColabFold

Non-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.


## 1 · Run the design campaign on your target (worked example: binder)

Honest campaign sizes (binder example): BindCraft 50–200; RFdiffusion-binder 500–1000 backbones →
ProteinMPNN → AF2-Multimer. We use a small **mock** count so the dry run is fast; scale up with the
real backend on A100. If your `DESIGN_TYPE` is an enzyme/antibody, swap in that family's generator —
the integration pattern is identical. **Mock numbers are SYNTHETIC.**

In [14]:
import campaign_tools as ct
import pandas as pd

DESIGN_TYPE = "binder"     # keep consistent with notebook 01 (your chosen type)
TARGET = "EXAMPLE_TARGET"
HOTSPOTS = ct.parse_hotspots("A56,A66,A115")   # EXAMPLE — replace with your verified residues

N_CAMPAIGN = 200           # -> 500-1000 backbones on A100; small batch / FreeBindCraft on T4
TOOL = "mock"              # -> "bindcraft" / "rfdiffusion" / "rfantibody" / "rfdiffusion2" on Colab (A100)

# Real call (Colab, A100): ct.generate_designs(TARGET, HOTSPOTS, n=N_CAMPAIGN, design_type=DESIGN_TYPE, tool="rfdiffusion")
pool = ct.generate_designs(TARGET, HOTSPOTS, n=N_CAMPAIGN, design_type=DESIGN_TYPE, tool=TOOL)
ct.score_designs(pool, tool=TOOL)              # AF2(-Multimer) -> pae_interaction, plddt, scrmsd, sc
campaign_df = ct.pool_to_df(pool)
campaign_df.to_csv("results/campaign_designs.csv", index=False)
print(f"campaign pool: {len(pool)} designs (design_type={DESIGN_TYPE}, tool={TOOL}; SYNTHETIC if mock)")
print("wrote results/campaign_designs.csv", campaign_df.shape)
campaign_df.head(4)[["design_id", "design_type", "scrmsd", "pae_interaction", "plddt", "rosetta_dG", "synthetic"]]

campaign pool: 200 designs (design_type=binder, tool=mock; SYNTHETIC if mock)
wrote results/campaign_designs.csv (200, 16)


,design_id,design_type,scrmsd,pae_interaction,plddt,rosetta_dG,synthetic
0,EXAMPLE_DATA_binder_0000,binder,1.03,19.0,83.0,-6.0,True
1,EXAMPLE_DATA_binder_0001,binder,3.76,12.0,96.0,-4.0,True
2,EXAMPLE_DATA_binder_0002,binder,1.84,10.0,84.0,-48.0,True
3,EXAMPLE_DATA_binder_0003,binder,1.45,17.0,75.0,-16.0,True


## 2 · Assemble the cohort feature table (EXAMPLE_DATA)

The ML success predictor learns from the **whole cohort**, not just your campaign. In a real run,
`build_cohort_table(csv_path="...")` loads the table your cohort assembled from Projects 01–24
outputs (sequences, in-silico metrics, any experimental labels). With no real cohort yet, it
**generates a deterministic `EXAMPLE_DATA` cohort** with a planted, *imperfect* feature→outcome
structure so the downstream ML runs anywhere.

We also **fold your own campaign designs into the table** (they get an in-silico-derived
`EXAMPLE_DATA` label here — replace with real experimental labels in notebook 05 when you have them).
Nothing here is a real experimental result.

In [15]:
import ml_predictor as ml

# Build (or, on a real run, LOAD) the cohort table. Multiple targets/design types => generalization test later.
cohort = ml.build_cohort_table(
    csv_path="data/cohort_design_outcomes.csv",   # if this exists (real cohort), it is LOADED; else synthesized
    n_per_target=120,
    targets=("binder", "binder", "enzyme", "antibody"),  # EXAMPLE cohort composition across Projects 01-24
    seed=0,
)
cohort.to_csv("results/cohort_table.csv", index=False)
print("cohort feature table:", cohort.shape)
print("design types present:", cohort["design_type"].value_counts().to_dict())
print("label origin:", cohort["label_origin"].value_counts().to_dict())
print("overall EXAMPLE_DATA success rate (synthetic!):", round(cohort["success"].mean(), 3))
print("\nEVERY row is source=EXAMPLE_DATA (synthetic) — never report as a real outcome.")
cohort.head(4)

cohort feature table: (480, 13)
design types present: {'binder': 240, 'enzyme': 120, 'antibody': 120}
label origin: {'EXAMPLE_DATA_SYNTHETIC': 480}
overall EXAMPLE_DATA success rate (synthetic!): 0.177

EVERY row is source=EXAMPLE_DATA (synthetic) — never report as a real outcome.


,design_id,target,design_type,scrmsd,plddt,pae_interaction,solubility,rosetta_dG,shape_complementarity,tm_to_pdb,source,label_origin,success
0,EXAMPLE_DATA_binder_0_0000,EXAMPLE_TARGET_binder_0,binder,3.120,71.2,3.78,-1.942,-11.90,0.875,0.614,EXAMPLE_DATA,EXAMPLE_DATA_SYNTHETIC,0
1,EXAMPLE_DATA_binder_0_0001,EXAMPLE_TARGET_binder_0,binder,4.253,89.7,3.05,1.001,-53.22,0.779,0.355,EXAMPLE_DATA,EXAMPLE_DATA_SYNTHETIC,0
2,EXAMPLE_DATA_binder_0_0002,EXAMPLE_TARGET_binder_0,binder,1.839,76.4,3.54,-1.565,-19.46,0.737,0.619,EXAMPLE_DATA,EXAMPLE_DATA_SYNTHETIC,0
3,EXAMPLE_DATA_binder_0_0003,EXAMPLE_TARGET_binder_0,binder,4.427,85.3,15.36,0.410,-34.39,0.470,0.683,EXAMPLE_DATA,EXAMPLE_DATA_SYNTHETIC,0


## 3 · (Real cohort) how the table is assembled from Projects 01–24

When you have a real cohort, the table is one row per design with: `design_id`, `design_type`,
`target`, the in-silico features (`scrmsd`, `plddt`, `pae_interaction`, `solubility`, `rosetta_dG`,
`shape_complementarity`, `tm_to_pdb`), and a `success` label. The label is **experimental where wet-lab
data exists** (`label_origin="experimental"`), otherwise an in-silico-derived proxy clearly flagged as
such. See `data/README.md` and `download_data.py` for what to assemble. **Never fabricate
experimental labels.**

In [16]:
# Scaffold for the REAL assembly (no-op here; the synthetic table above stands in for teaching):
#   frames = []
#   for proj in range(1, 25):
#       p = f"../../project_{proj:02d}_*/results/ranked.csv"   # each project's filtered design table
#       # read, normalize columns to FEATURE_COLUMNS, tag design_type/target, attach any experimental label
#   cohort_real = pd.concat(frames, ignore_index=True)
#   cohort_real.to_csv("data/cohort_design_outcomes.csv", index=False)
print("Real-cohort assembly is a scaffold; the EXAMPLE_DATA table above lets the ML notebook run anywhere.")
print("On a real run, write data/cohort_design_outcomes.csv and re-run cell 2 to LOAD it.")

Real-cohort assembly is a scaffold; the EXAMPLE_DATA table above lets the ML notebook run anywhere.
On a real run, write data/cohort_design_outcomes.csv and re-run cell 2 to LOAD it.


## D2 checklist
- [ ] Design campaign run at honest scale on your target (mock here; real backend + A100 on Colab).
- [ ] `results/campaign_designs.csv` written (one row per design, metrics parsed, SYNTHETIC if mock).
- [ ] Cohort feature table assembled/loaded → `results/cohort_table.csv` (every synthetic row `EXAMPLE_DATA`).
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured; 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on the campaign pool.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter on the campaign pool

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 25** you build `fp.Design` objects from your campaign pool (worked
example: `design_type="binder"`; use YOUR chosen type), call `fp.run_pipeline(...)`, and
`fp.report(...)` the survival funnel + ranked CSV. This is the **classical single-cutoff filter** —
the baseline the ML success predictor (notebook 04) must beat (D3 part 1).

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back — and the capstone's whole point is that notebook 04 *proposes* such an
> improvement (a learned predictor) from cohort data.

Run `00`-`02` first so `results/campaign_designs.csv` exists.

## Setup paths

In [17]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_25_capstone_dbtl/notebooks


## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. The
cutoffs depend on `design_type` (binder/enzyme/antibody/monomer).

In [18]:
import filtering_pipeline as fp
import pandas as pd

DESIGN_TYPE = "binder"     # keep consistent with notebooks 01-02 (your chosen type)
print("Loaded shared filtering_pipeline from:", fp.__file__)
print(f"{DESIGN_TYPE} cutoffs:", fp.DEFAULT_CUTOFFS[DESIGN_TYPE])

Loaded shared filtering_pipeline from: /home/user/biofx_python/denovo_protein_design_course/shared/filtering_pipeline.py
binder cutoffs: {'scrmsd': 2.5, 'plddt': 80, 'pae': 10, 'rosetta_dG': -30, 'sc': 0.6}


## Build `fp.Design` objects from the campaign pool

Map each campaign row onto `fp.Design`. The metrics drive the layers: `scrmsd`/`plddt`/
`pae_interaction` (Layer 1 self-consistency), `rosetta_dG`/`shape_complementarity`/`solubility`
(Layer 3 physics). We regenerate the pool deterministically if a fresh session lost the CSV. (Mock has
no independent orthogonal predictor, so we run Layers 1+3; on Colab add a second predictor for
Layer 2.)

In [19]:
import os
import campaign_tools as ct

if not os.path.exists("results/campaign_designs.csv"):
    HS = ct.parse_hotspots("A56,A66,A115")
    pool = ct.generate_designs("EXAMPLE_TARGET", HS, n=200, design_type=DESIGN_TYPE, tool="mock")
    ct.score_designs(pool, tool="mock")
    ct.pool_to_df(pool).to_csv("results/campaign_designs.csv", index=False)

camp = pd.read_csv("results/campaign_designs.csv")

def row_to_design(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type=DESIGN_TYPE,
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3), tm_to_pdb=r.get("tm_to_pdb"),
        catalytic_geom_rmsd=r.get("catalytic_geom_rmsd"),
        extra={"hotspot_overlap": r.get("hotspot_overlap"), "target": r.get("target")},
    )

designs = [row_to_design(r) for _, r in camp.iterrows()]
print(f"built {len(designs)} fp.Design objects (design_type={DESIGN_TYPE})")

built 200 fp.Design objects (design_type=binder)


## Run the pipeline + report (the classical single-cutoff filter)

`fp.run_pipeline(design_type=...)` applies the cutoffs in order and returns a ranked DataFrame with
survival counts in `df.attrs`. `fp.report(...)` prints the hit-rate accounting and draws the survival
funnel. Read the bars as a funnel: steep drops show which layer discriminates. **This survival/hit
rate is the BASELINE** that notebook 04's learned predictor is compared against.

In [20]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

df_ranked = fp.run_pipeline(designs, design_type=DESIGN_TYPE, use_layers=(1, 3))
df_ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(df_ranked, top_n=15, save_prefix="results/p25")
n = df_ranked.attrs["n_total"]; passed = int((df_ranked["layers_passed"] >= 3).sum())
print(f"\nclassical filter hit rate (all layers): {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")
print("saved results/p25_survival.png + results/p25_ranked.csv")
top

Total designs: 200
  L1 survivors: 34  (17.0%)
  L3 survivors: 14  (7.0%)



classical filter hit rate (all layers): 14/200 (7.0%)  [SYNTHETIC if mock]
saved results/p25_survival.png + results/p25_ranked.csv


,design_id,design_type,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG,tm_to_pdb
0,EXAMPLE_DATA_binder_0162,binder,3,4.5367,0.96,86.0,4.0,-36.0,0.68
1,EXAMPLE_DATA_binder_0197,binder,3,4.5133,0.97,87.0,5.0,-39.0,0.72
2,EXAMPLE_DATA_binder_0150,binder,3,4.4733,0.97,97.0,5.0,-32.0,0.82
3,EXAMPLE_DATA_binder_0074,binder,3,4.4367,1.06,96.0,4.0,-31.0,0.29
4,EXAMPLE_DATA_binder_0007,binder,3,4.1900,1.32,92.0,6.0,-42.0,0.49
5,EXAMPLE_DATA_binder_0063,binder,3,4.0200,1.19,99.0,9.0,-36.0,0.62
6,EXAMPLE_DATA_binder_0121,binder,3,4.0067,1.35,95.0,7.0,-37.0,0.52
7,EXAMPLE_DATA_binder_0183,binder,3,3.9167,1.22,82.0,10.0,-45.0,0.77
8,EXAMPLE_DATA_binder_0140,binder,3,3.7600,1.49,89.0,9.0,-43.0,0.29
9,EXAMPLE_DATA_binder_0002,binder,3,3.3767,1.84,84.0,10.0,-48.0,0.57


## Honest hit-rate accounting

Report `N passing all layers / N generated`, with the layer-by-layer survival counts — survival is
*enrichment*, not *correctness*. This is the classical-filter number the capstone tries to improve on
by **learning** a better triage rule from cohort data (notebook 04). Mock numbers are SYNTHETIC.

In [21]:
print("layers_passed distribution:", df_ranked["layers_passed"].value_counts().sort_index().to_dict())
print(f"all-layers survivors = {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")
print("\nRemember: a passing design is a HYPOTHESIS. The filter enriches; it does not guarantee.")

layers_passed distribution: {0: 166, 1: 20, 3: 14}
all-layers survivors = 14/200 (7.0%)  [SYNTHETIC if mock]

Remember: a passing design is a HYPOTHESIS. The filter enriches; it does not guarantee.


## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module (`design_type=` your type), not a one-off script.
- [ ] Survival-at-each-layer reported (funnel figure `results/p25_survival.png`).
- [ ] Honest classical-filter hit-rate accounting (N pass / N generated) — the baseline for notebook 04.
- [ ] Mapping assumptions (which fields → which `Design` attributes) written down.

**Next:** `04_validate.ipynb` — train the ML success predictor on the cohort table and benchmark it
against these single-metric cutoffs.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — train the ML success predictor + benchmark vs single-metric cutoffs

**Standard slot:** *validate (in silico).* **For Project 25 this is the core science (D3 part 2):**
train + cross-validate an ML model on the cohort feature table (`features → success`), report
**feature importance**, and benchmark its enrichment against the field's **single-metric cutoffs**
(scRMSD<2, pae_interaction<10, ...). This is the project's signature: closing the gap between
in-silico scores and experimental success by *learning* which features actually matter.

> **Honesty up front.** A real cohort is still a SMALL, BIASED, MULTI-TARGET dataset. We report
> **CV-AUC (mean ± std) and N**, never a single-split brag, and we keep the model small. The honest
> result is *"the model beats single-metric cutoffs by some margin"* — or, with small N, *"it doesn't
> yet"*. Both are valid capstone findings. **All numbers here are `EXAMPLE_DATA` (synthetic).**

Needs `results/cohort_table.csv` (from notebook 02). The ML part is **light / CPU-OK** — no GPU.

## Setup paths

In [22]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_25_capstone_dbtl/notebooks


## 1 · Load the cohort table + build (X, y)

`ml_predictor.build_cohort_table()` LOADS your real cohort table if present, else synthesizes the
deterministic `EXAMPLE_DATA` cohort. `features_and_label()` selects the complete feature columns and
drops incomplete rows. We print N and the class balance — the numbers that frame every claim below.

In [23]:
import ml_predictor as ml
import pandas as pd, os

if os.path.exists("results/cohort_table.csv"):
    cohort = pd.read_csv("results/cohort_table.csv")
else:
    cohort = ml.build_cohort_table(seed=0)   # deterministic EXAMPLE_DATA fallback
    cohort.to_csv("results/cohort_table.csv", index=False)

X, y, feat_cols = ml.features_and_label(cohort)
print("cohort table:", cohort.shape, "| features used:", feat_cols)
print(f"N = {len(y)}  successes = {int(y.sum())}  base rate = {y.mean():.3f}  [EXAMPLE_DATA if synthetic]")
print("label origin:", cohort["label_origin"].value_counts().to_dict())

cohort table: (480, 13) | features used: ['scrmsd', 'plddt', 'pae_interaction', 'solubility', 'rosetta_dG', 'shape_complementarity', 'tm_to_pdb']
N = 480  successes = 85  base rate = 0.177  [EXAMPLE_DATA if synthetic]
label origin: {'EXAMPLE_DATA_SYNTHETIC': 480}


## 2 · Train + cross-validate the success predictor

`train_success_predictor()` fits a small model with **StratifiedKFold cross-validation** and returns
CV ROC-AUC (mean ± std), N, and notes. Default is interpretable LogisticRegression; `model="rf"`
(RandomForest) captures non-linear feature interactions; `model="xgb"` uses XGBoost if installed and
**falls back to RandomForest with a note** otherwise (so this runs anywhere).

In [24]:
bundle = ml.train_success_predictor(X, y, feature_names=feat_cols, model="logreg", cv=5, seed=0)
print(f"model = {bundle['model_kind']}")
print(f"CV ROC-AUC = {bundle['cv_auc_mean']:.3f} +/- {bundle['cv_auc_std']:.3f}  "
      f"(folds={bundle['cv_folds']}, N={bundle['n']}, pos={bundle['n_pos']}, neg={bundle['n_neg']})")
for note in bundle["notes"]:
    print("  note:", note)

# Compare estimators honestly (RandomForest; XGBoost if available).
rf = ml.train_success_predictor(X, y, feature_names=feat_cols, model="rf", cv=5, seed=0)
xgb = ml.train_success_predictor(X, y, feature_names=feat_cols, model="xgb", cv=5, seed=0)
print(f"\nRandomForest CV-AUC = {rf['cv_auc_mean']:.3f} +/- {rf['cv_auc_std']:.3f}")
print(f"XGBoost/-fallback CV-AUC = {xgb['cv_auc_mean']:.3f} +/- {xgb['cv_auc_std']:.3f} ({xgb['model_kind']})")
print("\nAll AUCs are on EXAMPLE_DATA — they show the PLUMBING + honest reporting, not a real result.")

model = logreg
CV ROC-AUC = 0.878 +/- 0.032  (folds=5, N=480, pos=85, neg=395)
  note: Report CV-AUC (mean +/- std) and N. Train-fit is for importances only.



RandomForest CV-AUC = 0.831 +/- 0.046
XGBoost/-fallback CV-AUC = 0.831 +/- 0.046 (rf)

All AUCs are on EXAMPLE_DATA — they show the PLUMBING + honest reporting, not a real result.


## 3 · Feature importance — which features actually carry the signal?

For LogisticRegression these are **standardized coefficients** (sign = direction: does a higher value
raise or lower predicted success?). For tree models they are gain/impurity importances (magnitude
only). This is the "which filters matter" answer the cohort cares about — read it against the
field-standard cutoffs.

In [25]:
imp = ml.feature_importance(bundle)
print("feature importance (standardized LogReg coefficients; EXAMPLE_DATA):")
print(imp.to_string(index=False))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(imp["feature"][::-1], imp["importance"][::-1])
ax.set_xlabel("|importance|  (EXAMPLE_DATA)")
ax.set_title("Success-predictor feature importance")
plt.tight_layout(); plt.savefig("results/p25_feature_importance.png", dpi=150); plt.show()
print("saved results/p25_feature_importance.png")

feature importance (standardized LogReg coefficients; EXAMPLE_DATA):
              feature  importance          direction
               scrmsd      1.5379 - (lowers success)
           rosetta_dG      0.8901 - (lowers success)
      pae_interaction      0.8536 - (lowers success)
           solubility      0.4786 + (raises success)
                plddt      0.4400 + (raises success)
shape_complementarity      0.2371 + (raises success)
            tm_to_pdb      0.0009 - (lowers success)


saved results/p25_feature_importance.png


## 4 · The headline benchmark — ML vs single-metric cutoffs (enrichment)

`compare_to_single_metric_cutoffs()` reports, for each field-standard cutoff **and** for the ML model
selecting a comparable fraction of designs: how many it selects, the **precision** (success rate among
selected), the **enrichment** (precision / base rate), and the **recall** (fraction of all successes
captured). The capstone question: *does the learned predictor enrich better — and/or recall more
successes — at a comparable selection size than any single metric?* Report what you find.

In [26]:
cmp = ml.compare_to_single_metric_cutoffs(cohort, bundle)
print(f"base success rate = {cmp.attrs.get('base_rate')}  (N={cmp.attrs.get('n_total')}, "
      f"successes={cmp.attrs.get('n_success')})  [EXAMPLE_DATA]")
print()
print(cmp.to_string(index=False))
cmp.to_csv("results/p25_enrichment_vs_cutoffs.csv", index=False)

best_single = cmp[cmp["kind"] == "single-metric"]["enrichment"].max()
ml_row = cmp[cmp["kind"] == "ML-composite"]
ml_enr = float(ml_row["enrichment"].iloc[0]) if len(ml_row) else float("nan")
verdict = ("BEATS" if ml_enr >= best_single else "does NOT beat")
print(f"\nVERDICT (EXAMPLE_DATA): ML enrichment {ml_enr} {verdict} best single-metric enrichment {best_single}.")
print("On a real cohort this verdict may differ — REPORT IT HONESTLY, with N and CV-AUC.")

base success rate = 0.1771  (N=480, successes=85)  [EXAMPLE_DATA]

                        selector          kind  n_selected  frac_selected  precision  enrichment  recall
                   scrmsd <= 2.0 single-metric         155          0.323      0.361        2.04   0.659
                   plddt >= 80.0 single-metric         208          0.433      0.231        1.30   0.565
         pae_interaction <= 10.0 single-metric         161          0.335      0.292        1.65   0.553
             rosetta_dG <= -30.0 single-metric         192          0.400      0.276        1.56   0.624
    shape_complementarity >= 0.6 single-metric         265          0.552      0.211        1.19   0.659
ML model (top 40% by P(success))  ML-composite         192          0.400      0.385        2.18   0.871

VERDICT (EXAMPLE_DATA): ML enrichment 2.18 BEATS best single-metric enrichment 2.04.
On a real cohort this verdict may differ — REPORT IT HONESTLY, with N and CV-AUC.


In [27]:
# Enrichment bar chart (single metrics vs the ML composite). EXAMPLE_DATA.
fig, ax = plt.subplots(figsize=(7, 3.4))
colors = ["#888" if k == "single-metric" else "#1f77b4" for k in cmp["kind"]]
ax.bar(range(len(cmp)), cmp["enrichment"], color=colors)
ax.axhline(1.0, color="k", lw=0.8, ls="--", label="no enrichment (=base rate)")
ax.set_xticks(range(len(cmp)))
ax.set_xticklabels(cmp["selector"], rotation=40, ha="right", fontsize=7)
ax.set_ylabel("enrichment (precision / base rate)")
ax.set_title("ML predictor vs single-metric cutoffs (EXAMPLE_DATA)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("results/p25_enrichment.png", dpi=150); plt.show()
print("saved results/p25_enrichment.png")

saved results/p25_enrichment.png


## 5 · Cross-target generalization `[extension]`

A predictor that only works on the design type it was trained on is weak. Hold out one design type,
train on the rest, and test on the held-out type — the honest test of whether the learned rule
**generalizes** across the cohort. With small per-type N this is noisy; report the caveat.

In [28]:
import numpy as np
print("leave-one-design-type-out CV-AUC (EXAMPLE_DATA; small-N, noisy):")
from sklearn.metrics import roc_auc_score
for held in sorted(cohort["design_type"].unique()):
    tr = cohort[cohort["design_type"] != held]
    te = cohort[cohort["design_type"] == held]
    Xtr, ytr, cols = ml.features_and_label(tr)
    b = ml.train_success_predictor(Xtr, ytr, feature_names=cols, model="logreg", seed=0)
    if b["estimator"] is None:
        print(f"  hold out {held:9s}: insufficient data"); continue
    Xte = te.dropna(subset=cols + ["success"])[cols].to_numpy(float)
    yte = te.dropna(subset=cols + ["success"])["success"].to_numpy(int)
    try:
        p = b["estimator"].predict_proba(Xte)[:, 1]
        auc = roc_auc_score(yte, p) if len(set(yte)) > 1 else float("nan")
    except Exception as e:  # noqa: BLE001
        auc = float("nan")
    print(f"  hold out {held:9s}: test ROC-AUC = {auc:.3f}  (test N={len(yte)})")
print("\nGeneralization across targets is the hard part; small per-type N makes this noisy (EXAMPLE_DATA).")

leave-one-design-type-out CV-AUC (EXAMPLE_DATA; small-N, noisy):
  hold out antibody : test ROC-AUC = 0.904  (test N=120)
  hold out binder   : test ROC-AUC = 0.877  (test N=240)
  hold out enzyme   : test ROC-AUC = 0.846  (test N=120)

Generalization across targets is the hard part; small per-type N makes this noisy (EXAMPLE_DATA).


## D3 (part 2) checklist
- [ ] Cohort table loaded; **N + class balance** printed (frames every claim).
- [ ] Success predictor trained with **cross-validation**; CV-AUC (mean ± std) + N reported (not a single split).
- [ ] **Feature importance** computed + figure (`results/p25_feature_importance.png`).
- [ ] **Enrichment vs single-metric cutoffs** table + figure; honest VERDICT (beats / doesn't, by how much).
- [ ] (extension) Cross-target generalization (leave-one-type-out) reported with the small-N caveat.
- [ ] Every number flagged `EXAMPLE_DATA`; overfitting / N caveats stated.

**Next:** `05_validation_plan.ipynb` — execute-or-plan validation, integrate labels, honest hit-rate +
failure forensics + active-learning loop.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validate-or-plan + honest hit-rate + failure forensics + active-learning loop

**Standard slot:** *validation plan.* **For Project 25 this closes the DBTL loop (D4/D5):**
execute *or fully plan* experimental validation of the top campaign designs (with controls), integrate
any experimental labels back into the success predictor, deliver an **honest hit-rate + failure-
forensics** analysis, a cohort-wide **"lessons learned"** synthesis, and design an **active-learning
loop** (which design to test next) `[stretch]`.

> **No fabricated experimental results.** A design is a *hypothesis* until measured. Any labels you
> integrate must be REAL wet-lab outcomes; absent those, the analysis runs on `EXAMPLE_DATA` and is
> clearly labeled. **Synthesis screening + institutional biosafety/ethics approval are required for any
> real wet-lab work** (see Responsible Research).

Run `00`–`04` first.

## Setup paths

In [29]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_25_capstone_dbtl/notebooks


## 1 · The experimental validation plan (controls are mandatory)

Write the costed, controlled plan for the **top candidates** from notebook 03/04. The assay depends on
your `design_type`: binders → **SPR/BLI** (K_D/kinetics) + a competition assay; enzymes → an **activity
assay** (kcat/KM) + a dead-mutant control; antibodies → binding + developability. The three controls
are non-negotiable.

In [30]:
DESIGN_TYPE = "binder"   # your chosen type
PLAN = {
    "binder":   dict(assay="SPR or BLI vs immobilized target (K_D, kon/koff) + competition assay",
                     expression="E. coli BL21(DE3), His-tagged, 16-18 C overnight; target reagent often mammalian/commercial"),
    "antibody": dict(assay="SPR/BLI binding + developability (SEC, Tm/DSF, poly-specificity)",
                     expression="mammalian (Expi293) or yeast display for screening"),
    "enzyme":   dict(assay="activity assay (kcat/KM) on the substrate + catalytic dead-mutant control",
                     expression="E. coli; purify; confirm fold by CD/SEC before activity"),
    "monomer":  dict(assay="go/no-go: express -> SDS-PAGE -> SEC; fold by CD/DSF",
                     expression="E. coli BL21(DE3), 16-18 C overnight"),
}[DESIGN_TYPE]

CONTROLS = {
    "positive": "a known-good binder/enzyme/natural protein — confirms the assay + reagent are active",
    "negative_scrambled": "YOUR OWN top design with the interface scrambled / catalytic residue mutated — must LOSE activity",
    "negative_unrelated": "an unrelated protein of similar size that should not bind/act",
}
print("design_type:", DESIGN_TYPE)
print("assay      :", PLAN["assay"])
print("expression :", PLAN["expression"])
print("controls (mandatory):")
for k, v in CONTROLS.items():
    print(f"  {k}: {v}")
print("\nCosted reagent list + timeline go in the written plan (D4). Synthesis via an IGSC-screening provider.")

design_type: binder
assay      : SPR or BLI vs immobilized target (K_D, kon/koff) + competition assay
expression : E. coli BL21(DE3), His-tagged, 16-18 C overnight; target reagent often mammalian/commercial
controls (mandatory):
  positive: a known-good binder/enzyme/natural protein — confirms the assay + reagent are active
  negative_scrambled: YOUR OWN top design with the interface scrambled / catalytic residue mutated — must LOSE activity
  negative_unrelated: an unrelated protein of similar size that should not bind/act

Costed reagent list + timeline go in the written plan (D4). Synthesis via an IGSC-screening provider.


## 2 · Integrate experimental labels (when you have them) — no fabrication

When real wet-lab outcomes come back, put them in a small CSV (`design_id, success`) and re-train the
predictor with them via `build_cohort_table(experimental_labels=...)`. The touched rows get
`label_origin="experimental"`. Below we DEMONSTRATE the wiring on an **EXAMPLE_DATA** label set (a
deterministic synthetic stand-in) — on a real run, replace it with your measured outcomes. We NEVER
fabricate K_D/kcat values.

In [31]:
import ml_predictor as ml
import pandas as pd

cohort = pd.read_csv("results/cohort_table.csv") if __import__("os").path.exists("results/cohort_table.csv")     else ml.build_cohort_table(seed=0)

# EXAMPLE_DATA stand-in for "experimental labels just came back for a handful of designs".
# On a real run: exp = pd.read_csv("data/experimental_labels.csv")  # columns: design_id, success
exp_demo = cohort.sample(min(15, len(cohort)), random_state=0)[["design_id"]].copy()
exp_demo["success"] = (ml.build_cohort_table(seed=7)["success"].head(len(exp_demo)).values)  # synthetic placeholder
exp_demo["NOTE"] = "EXAMPLE_DATA — replace with REAL measured outcomes; never fabricate"

relabeled = ml._attach_experimental(cohort.copy(), exp_demo[["design_id", "success"]])
print("rows now flagged experimental (EXAMPLE_DATA demo):",
      int((relabeled["label_origin"] == "experimental").sum()))
X, y, cols = ml.features_and_label(relabeled)
b2 = ml.train_success_predictor(X, y, feature_names=cols, model="logreg", seed=0)
print(f"re-trained CV-AUC with integrated labels = {b2['cv_auc_mean']:.3f} +/- {b2['cv_auc_std']:.3f} (EXAMPLE_DATA)")
print("On a real run this is where the loop CLOSES: measured outcomes improve the shared filter.")

rows now flagged experimental (EXAMPLE_DATA demo): 15
re-trained CV-AUC with integrated labels = 0.868 +/- 0.040 (EXAMPLE_DATA)
On a real run this is where the loop CLOSES: measured outcomes improve the shared filter.


## 3 · Honest hit-rate + failure forensics

Report the classical-filter hit rate (notebook 03) AND, where labels exist, the **true** success rate
among the designs you actually tested. Then do **failure forensics**: of the designs that passed the
in-silico filter but FAILED experimentally (false positives), what do they have in common? This is the
single most useful output for the next cohort — it tells them which "confident" designs to distrust.

In [32]:
import numpy as np
# Among labelled designs, contrast feature distributions of experimental success vs failure.
lab = relabeled.copy()
lab = lab[lab["label_origin"] == "experimental"] if (lab["label_origin"] == "experimental").any() else relabeled
print("failure-forensics: mean feature values by outcome (EXAMPLE_DATA):")
forensics = lab.groupby("success")[ml.FEATURE_COLUMNS].mean(numeric_only=True).round(2)
print(forensics.T.rename(columns={0: "FAIL_mean", 1: "SUCCESS_mean"}).to_string())

# "False positives": passed the classical scRMSD+pae filter but labelled failure.
fp_mask = (lab["scrmsd"] <= 2.0) & (lab["pae_interaction"] <= 10.0) & (lab["success"] == 0)
print(f"\n'confident-but-failed' designs (scrmsd<=2 & pae<=10 yet success=0): {int(fp_mask.sum())}")
print("These false positives are the failure-forensics target — what do they share? (EXAMPLE_DATA)")

failure-forensics: mean feature values by outcome (EXAMPLE_DATA):
success                FAIL_mean  SUCCESS_mean
scrmsd                      2.63          0.64
plddt                      76.59         63.60
pae_interaction            12.83         13.08
solubility                  0.13         -1.43
rosetta_dG                -31.37        -28.51
shape_complementarity       0.67          0.77
tm_to_pdb                   0.54          0.80

'confident-but-failed' designs (scrmsd<=2 & pae<=10 yet success=0): 3
These false positives are the failure-forensics target — what do they share? (EXAMPLE_DATA)


## 4 · Active-learning loop — which design to test next? `[stretch]`

Wet-lab tests are expensive, so spend them where they are most informative. Two classic acquisition
rules on the predictor's `P(success)`:
- **Exploitation:** test the highest-`P(success)` designs (most likely to work).
- **Exploration / uncertainty:** test designs where `P(success) ≈ 0.5` (the model is least sure — each
  label teaches it the most).

A real loop alternates: test a batch → add labels → re-train → re-rank. Below we rank the campaign
designs by both rules so you can pick the next batch. **EXAMPLE_DATA.**

In [33]:
import os
camp = pd.read_csv("results/campaign_designs.csv") if os.path.exists("results/campaign_designs.csv") else None
bundle = ml.train_success_predictor(*ml.features_and_label(cohort)[:2],
                                    feature_names=ml.features_and_label(cohort)[2], model="logreg", seed=0)
if camp is not None and bundle["estimator"] is not None:
    cols = bundle["feature_names"]
    use = camp.dropna(subset=[c for c in cols if c in camp.columns]).copy()
    have = [c for c in cols if c in use.columns]
    if len(have) == len(cols) and len(use):
        proba = bundle["estimator"].predict_proba(use[cols].to_numpy(float))[:, 1]
        use["p_success"] = proba
        use["uncertainty"] = 1.0 - (use["p_success"] - 0.5).abs() * 2.0   # 1 at p=0.5, 0 at p=0/1
        exploit = use.sort_values("p_success", ascending=False).head(5)
        explore = use.sort_values("uncertainty", ascending=False).head(5)
        print("ACTIVE LEARNING — next batch to test (EXAMPLE_DATA):")
        print("\n exploitation (highest P(success)):")
        print(exploit[["design_id", "p_success"]].to_string(index=False))
        print("\n exploration (most uncertain, P~0.5):")
        print(explore[["design_id", "p_success", "uncertainty"]].to_string(index=False))
        use.to_csv("results/p25_active_learning_ranking.csv", index=False)
        print("\nsaved results/p25_active_learning_ranking.csv")
    else:
        print("Active-learning scaffold: campaign features incomplete for the model's columns.")
else:
    print("Active-learning scaffold — needs results/campaign_designs.csv + a trained model.")

ACTIVE LEARNING — next batch to test (EXAMPLE_DATA):

 exploitation (highest P(success)):
               design_id  p_success
EXAMPLE_DATA_binder_0007   0.990384
EXAMPLE_DATA_binder_0074   0.990335
EXAMPLE_DATA_binder_0197   0.989180
EXAMPLE_DATA_binder_0149   0.988691
EXAMPLE_DATA_binder_0150   0.985996

 exploration (most uncertain, P~0.5):
               design_id  p_success  uncertainty
EXAMPLE_DATA_binder_0178   0.494188     0.988377
EXAMPLE_DATA_binder_0044   0.490724     0.981448
EXAMPLE_DATA_binder_0142   0.490122     0.980245
EXAMPLE_DATA_binder_0089   0.510104     0.979792
EXAMPLE_DATA_binder_0038   0.515779     0.968441

saved results/p25_active_learning_ranking.csv


## 5 · Cohort-wide "lessons learned" synthesis (D5)

The capstone's final synthesis (write it up in the thesis, seed it here):
- **Which features actually predicted success** across the cohort (notebook 04 importances) — and
  which "trusted" single metrics were weak.
- **The honest hit rate** of the classical filter vs the learned predictor (enrichment + recall).
- **Failure forensics:** the signature of confident-but-wrong designs.
- **A concrete proposal to improve `shared/filtering_pipeline.py`** (new `DEFAULT_CUTOFFS` or a learned
  composite score) — pull-requested back for Projects 01–24's successors.
- **The active-learning recommendation:** the next batch to test.

In [34]:
LESSONS = """# Capstone lessons-learned (fill from YOUR results; EXAMPLE_DATA placeholders)

1. Best single predictor on the cohort: <feature> (enrichment <x>, recall <y>).
2. Learned composite CV-AUC = <a +/- b>, N=<n>; enrichment <e> vs best single <s> -> <beats/does not>.
3. Confident-but-failed signature (failure forensics): <what false positives share>.
4. Proposed shared-filter change (PR): <new cutoffs / add learned score>; evidence: CV-AUC + N.
5. Active-learning next batch: <design_ids> (exploit) + <design_ids> (explore).
6. Caveats: small/biased N; multi-target; in-silico labels where experimental are missing.
"""
open("results/p25_lessons_learned.md", "w").write(LESSONS)
print("wrote results/p25_lessons_learned.md (template — fill with YOUR results).")
print("\nReminder: a design is a HYPOTHESIS until measured; report the hit rate, not the cherry.")

wrote results/p25_lessons_learned.md (template — fill with YOUR results).

Reminder: a design is a HYPOTHESIS until measured; report the hit rate, not the cherry.


## D4 / D5 checklist
- [ ] Costed, controlled **validation plan** (assay for your design_type; positive + scrambled + unrelated controls).
- [ ] Any **real experimental labels integrated** (`label_origin="experimental"`); model re-trained — no fabrication.
- [ ] **Honest hit-rate** (classical filter vs learned predictor) + **failure forensics** (false-positive signature).
- [ ] **Active-learning** next-batch ranking (exploit + explore) `[stretch]`.
- [ ] Cohort-wide **lessons-learned** synthesis + a concrete `filtering_pipeline.py` improvement proposal.
- [ ] D★: the improved success-predictor module (`scripts/ml_predictor.py`) + honest hit-rate + lessons.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — and the cohort's shared filter is now better because of your learned predictor.